In [1]:
#| default_exp io.export_data

In [2]:
#| export
from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd

from myproj.io.import_data import PROJECT_ROOT

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

logger = logging.getLogger("myproj.io")

# Daten exportieren

In [3]:
from myproj.pipeline import run_import, run_pre_filter, run_cleaning, run_post_filter, run_transform, run_join

emdat_raw, sea_level_raw = run_import()
emdat, sea_level = run_pre_filter(emdat_raw, sea_level_raw)
emdat, sea_level = run_cleaning(emdat, sea_level)
emdat_flood = run_post_filter(emdat)
emdat_flood = run_transform(emdat_flood)
flood_linked = run_join(emdat_flood, sea_level)

print(f"flood_linked: {flood_linked.shape[0]} Ereignisse, {flood_linked.shape[1]} Spalten")

2026-05-13 22:07:25 | myproj.pipeline      | INFO     | run_import | start
2026-05-13 22:07:25 | myproj.io            | INFO     | Lade Rohdatei: public_emdat_1991_2024.xlsx
2026-05-13 22:07:29 | myproj.io            | INFO     | load_raw_data | file: public_emdat_1991_2024.xlsx | rows: 20657 | cols: 47
2026-05-13 22:07:29 | myproj.io            | INFO     | Lade Rohdatei: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc
2026-05-13 22:07:29 | myproj.io            | INFO     | load_raw_data | file: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc | variables: 2 | dimensions: {'time': 9405}
2026-05-13 22:07:29 | myproj.pipeline      | INFO     | run_import | done
2026-05-13 22:07:29 | myproj.pipeline      | INFO     | run_pre_filter | start
2026-05-13 22:07:29 | myproj.transform     | INFO     | select_columns | cols: 47 → 18
2026-05-13 22:07:29 | myproj.transform     | INFO     | filter_to_sea_level_coverage | rows: 20657 → 16699 | removed before: 3

flood_linked: 292 Ereignisse, 28 Spalten


## Finaler Datensatz: `flood_linked`

In [6]:
print(f"flood_linked: {flood_linked.shape[0]} Ereignisse, {flood_linked.shape[1]} Spalten")
print(f"Fehlende sea_level_at_start: {flood_linked['sea_level_at_start'].isna().sum()}")
display(flood_linked.head(10))

flood_linked: 292 Ereignisse, 28 Spalten
Fehlende sea_level_at_start: 0


,DisNo.,ISO,Country,Region,Disaster Subtype,Origin,Latitude,Longitude,Total Affected,Total Deaths,...,sea_trend_at_start,sea_level_at_end,mean_sea_level_while_disaster,sea_level_lag_1d,sea_level_lag_2d,sea_level_lag_3d,sea_level_lag_4d,sea_level_lag_5d,sea_level_at_start_z,mean_sea_level_while_disaster_z
0,1999-0450-FRA,FRA,France,Europe,Riverine flood,Brief torrential rain,NaN,NaN,3005.0,36.0,...,2.293326,1.229269,1.271478,1.342794,1.371823,1.401049,1.430440,1.459967,-1.411555,-1.424464
1,2000-0708-GRC,GRC,Greece,Europe,Flood (General),Extreme rain,NaN,NaN,600.0,NaN,...,2.468741,2.305542,2.305542,2.235074,2.165319,2.096304,2.028052,1.960584,-1.110504,-1.110504
2,2023-0866-FRA,FRA,France,Europe,Flood (General),Heavy rains,NaN,NaN,6050.0,1.0,...,10.965430,14.527058,14.524420,14.518651,14.515857,14.512674,14.509099,14.505121,2.598336,2.599355
3,2022-0825-HRV,HRV,Croatia,Europe,Flood (General),NaN,NaN,NaN,NaN,NaN,...,10.400740,12.033220,12.033220,12.048471,12.062099,12.074087,12.084422,12.093092,1.842984,1.842984
4,2022-0825-BIH,BIH,Bosnia and Herzegovina,Europe,Flood (General),NaN,NaN,NaN,3000.0,1.0,...,10.400740,11.764241,11.912309,12.048471,12.062099,12.074087,12.084422,12.093092,1.842984,1.806273
5,2014-0164-BIH,BIH,Bosnia and Herzegovina,Europe,Riverine flood,Heavy rains,NaN,NaN,1000000.0,25.0,...,6.427430,5.778097,5.671382,5.556266,5.539779,5.523946,5.508777,5.494277,-0.118327,-0.088578
6,2001-0642-GRC,GRC,Greece,Europe,Flash flood,Heavy rain,NaN,NaN,600.0,NaN,...,2.680990,0.013958,0.013958,0.038181,0.063944,0.091210,0.119938,0.150084,-1.806268,-1.806268
7,2001-0802-BIH,BIH,Bosnia and Herzegovina,Europe,Riverine flood,Heavy rain,NaN,NaN,9000.0,NaN,...,2.591551,1.331541,1.329622,1.324738,1.322653,1.321453,1.321145,1.321733,-1.407393,-1.406810
8,2019-0194-BIH,BIH,Bosnia and Herzegovina,Europe,Riverine flood,Heavy rain,NaN,NaN,600.0,1.0,...,8.603725,6.793387,6.788072,6.772382,6.762283,6.752481,6.742994,6.733842,0.248854,0.250468
9,2002-0764-GRC,GRC,Greece,Europe,Riverine flood,Heavy rain,NaN,NaN,NaN,NaN,...,2.897837,5.076468,5.126674,5.195095,5.214692,5.232920,5.249783,5.265288,-0.239554,-0.253961


## Daten exportieren
Speichert verarbeitete Datensätze in `data/processed/` als Parquet.

In [ ]:
#| export
def save_processed_data(df: pd.DataFrame, filename: str) -> Path:
    """Speichert einen DataFrame als Parquet in data/processed/."""
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    path = PROCESSED_DIR / filename
    df.to_parquet(path, index=False)
    logger.info(
        "save_processed_data | file: %s | rows: %d | cols: %d",
        filename, len(df), len(df.columns),
    )
    return path

In [5]:
save_processed_data(flood_linked, "flood_linked.parquet")

2026-05-13 22:07:29 | myproj.io            | INFO     | save_processed_data | file: flood_linked.parquet | rows: 292 | cols: 28


PosixPath('/Users/damianszedalik/Documents/01_Coding/02_FHNW/05_Datenverarbeitung_Infrastruktur/DAW_FS26/data/processed/flood_linked.parquet')